# AgentsVille AI Trip Planner – Test Scenarios

This notebook validates the full pipeline across six diverse traveler profiles.

Each scenario:
1. Defines unique `VacationInfo` parameters
2. Runs the full pipeline (ItineraryAgent → Evaluation → Revision → Summary)
3. Validates all evaluations pass
4. Displays results and reasoning log
5. Extracts key metrics (iterations needed, budget utilisation, activity variety)


In [ ]:
import json
import os
from pathlib import Path

from openai import OpenAI

from project_lib import (
    VacationInfo,
    TravelPlan,
    get_weather_forecast,
    get_available_activities,
    run_evals,
    ItineraryAgent,
    ItineraryRevisionAgent,
    generate_trip_summary,
    print_itinerary,
    print_eval_results,
)

print("✅ Imports successful")

In [ ]:
# ── Configure your OpenAI API key ────────────────────────────────────────────
# Option A: set it here directly (not recommended for shared notebooks)
# os.environ["OPENAI_API_KEY"] = "sk-..."
#
# Option B: export it in your shell before launching Jupyter
#   export OPENAI_API_KEY="sk-..."

client = OpenAI()  # reads OPENAI_API_KEY from the environment

# Model configuration
MAIN_MODEL = "gpt-4o"
EVAL_MODEL = "gpt-4o-mini"

print(f"✅ OpenAI client ready  |  main model: {MAIN_MODEL}  |  eval model: {EVAL_MODEL}")

In [ ]:
# ── Shared helper ─────────────────────────────────────────────────────────────

def run_scenario(scenario_name: str, vacation_info: VacationInfo) -> dict:
    """Run the full pipeline for a single test scenario.

    Returns a metrics dict with keys:
        scenario, passed, iterations, total_cost, budget,
        budget_utilisation_pct, unique_activities, days
    """
    print(f"\n{'='*65}")
    print(f"  {scenario_name}")
    print(f"{'='*65}")
    print(f"  Destination : {vacation_info.destination}")
    print(f"  Dates       : {vacation_info.start_date} → {vacation_info.end_date}")
    print(f"  Budget      : ${vacation_info.budget:.2f}")
    print(f"  Interests   : {', '.join(vacation_info.interests)}")
    if vacation_info.constraints:
        print(f"  Constraints : {', '.join(vacation_info.constraints)}")

    # Step 1 – gather data
    weather_data = get_weather_forecast(vacation_info)
    available_activities = get_available_activities(vacation_info, weather_data)

    # Step 2 – generate initial plan
    print("\n⏳ Generating initial itinerary…")
    agent = ItineraryAgent(client, model=MAIN_MODEL)
    plan = agent.generate(vacation_info, weather_data, available_activities)

    # Step 3 – evaluate
    print("\n📊 Running initial evaluations…")
    eval_results = run_evals(plan, vacation_info, weather_data, available_activities,
                             client, model=EVAL_MODEL)
    print_eval_results(eval_results)

    # Step 4 – revise if needed
    revision_agent = ItineraryRevisionAgent(client, model=MAIN_MODEL)
    if not eval_results.get("all_passed", False):
        print("\n🔄 Revision required – starting ReAct loop…")
        plan = revision_agent.revise(
            plan, vacation_info, weather_data, available_activities,
            eval_model=EVAL_MODEL,
        )
        # Re-evaluate after revision
        eval_results = run_evals(plan, vacation_info, weather_data, available_activities,
                                 client, model=EVAL_MODEL)
        print("\n📊 Post-revision evaluation results:")
        print_eval_results(eval_results)
    else:
        print("\n✅ No revision needed.")

    # Step 5 – generate summary
    print("\n📝 Generating trip summary…")
    plan.summary = generate_trip_summary(plan, vacation_info, client, model=MAIN_MODEL)
    print_itinerary(plan)

    # Step 6 – metrics
    unique_acts = {a.name for day in plan.days for a in day.activities}
    iterations = len([e for e in revision_agent.reasoning_log if e.get("type") == "action"])
    budget_pct = round(plan.total_cost / vacation_info.budget * 100, 1)
    passed = eval_results.get("all_passed", False)

    metrics = {
        "scenario": scenario_name,
        "passed": passed,
        "iterations": iterations,
        "total_cost": plan.total_cost,
        "budget": vacation_info.budget,
        "budget_utilisation_pct": budget_pct,
        "unique_activities": len(unique_acts),
        "days": len(plan.days),
    }
    print(f"\n📈 Metrics: {metrics}")
    assert passed, f"❌ SCENARIO FAILED: {scenario_name} – not all checks pass"
    print(f"\n✅ {scenario_name} PASSED")
    return metrics

# Collect all scenario metrics for the summary table
all_metrics = []

---
## Scenario 1 – Budget-Conscious Traveler

**Profile:** Traveler on a tight budget of $100.  
**Goal:** Verify the planner selects low-cost activities and stays within budget.


In [ ]:
vacation_info_s1 = VacationInfo(
    destination="AgentsVille",
    start_date="2025-06-10",
    end_date="2025-06-11",   # 2-day trip to keep cost low
    interests=["sightseeing", "food"],
    budget=100.0,
    constraints=["budget-conscious"],
)

metrics_s1 = run_scenario("Scenario 1 – Budget-Conscious Traveler", vacation_info_s1)
all_metrics.append(metrics_s1)

---
## Scenario 2 – Adventure Seekers

**Profile:** Travelers who love outdoor activities, hiking, and sports.  
**Goal:** Verify the planner selects weather-appropriate outdoor activities.


In [ ]:
vacation_info_s2 = VacationInfo(
    destination="AgentsVille",
    start_date="2025-07-01",
    end_date="2025-07-03",
    interests=["outdoor", "sports", "hiking"],
    budget=500.0,
    constraints=[],
)

metrics_s2 = run_scenario("Scenario 2 – Adventure Seekers", vacation_info_s2)
all_metrics.append(metrics_s2)

---
## Scenario 3 – Culture Enthusiasts

**Profile:** Travelers interested in art, theatre, and museums.  
**Goal:** Verify the planner prioritises culture-category activities.


In [ ]:
vacation_info_s3 = VacationInfo(
    destination="AgentsVille",
    start_date="2025-08-05",
    end_date="2025-08-07",
    interests=["art", "theatre", "museums", "culture"],
    budget=400.0,
    constraints=[],
)

metrics_s3 = run_scenario("Scenario 3 – Culture Enthusiasts", vacation_info_s3)
all_metrics.append(metrics_s3)

---
## Scenario 4 – Food Lovers

**Profile:** Travelers passionate about cooking, restaurant tours, and food markets.  
**Goal:** Verify the planner selects food-category activities and stays within budget.


In [ ]:
vacation_info_s4 = VacationInfo(
    destination="AgentsVille",
    start_date="2025-09-12",
    end_date="2025-09-14",
    interests=["food", "cooking", "restaurants", "farmers markets"],
    budget=450.0,
    constraints=["vegetarian"],
)

metrics_s4 = run_scenario("Scenario 4 – Food Lovers", vacation_info_s4)
all_metrics.append(metrics_s4)

---
## Scenario 5 – Extended Trip (5+ Days – Scalability Test)

**Profile:** Travelers planning a longer vacation.  
**Goal:** Verify the planner handles 5+ days without degrading quality.


In [ ]:
vacation_info_s5 = VacationInfo(
    destination="AgentsVille",
    start_date="2025-10-01",
    end_date="2025-10-06",   # 6-day trip
    interests=["culture", "food", "outdoor", "entertainment"],
    budget=1200.0,
    constraints=[],
)

metrics_s5 = run_scenario("Scenario 5 – Extended Trip (6 days)", vacation_info_s5)
all_metrics.append(metrics_s5)

---
## Scenario 6 – Mixed Interests with Weather Challenges

**Profile:** Travelers with broad interests during a period of mixed (rainy) weather.  
**Goal:** Verify the planner correctly handles rainy days by selecting only indoor activities.


In [ ]:
# Dates chosen to hit rainy days based on the deterministic weather simulation:
#   (day * 3 + month * 7) % 8 – index 6 in pool = "rainy"
vacation_info_s6 = VacationInfo(
    destination="AgentsVille",
    start_date="2025-11-10",
    end_date="2025-11-12",
    interests=["culture", "food", "entertainment", "wellness"],
    budget=600.0,
    constraints=["accessibility"],
)

metrics_s6 = run_scenario("Scenario 6 – Mixed Interests with Weather Challenges", vacation_info_s6)
all_metrics.append(metrics_s6)

---
## Test Results Summary


In [ ]:
print("\n" + "="*75)
print("  TEST SCENARIOS SUMMARY")
print("="*75)
print(f"{'Scenario':<42} {'Pass':>4} {'Days':>4} {'Cost':>8} {'Budget%':>8} {'Itr':>4} {'UActv':>6}")
print("-"*75)
for m in all_metrics:
    name = m["scenario"].split(" – ", 1)[1] if " – " in m["scenario"] else m["scenario"]
    passed_icon = "✅" if m["passed"] else "❌"
    print(
        f"{name:<42} {passed_icon:>4} {m['days']:>4} "
        f"${m['total_cost']:>7.2f} {m['budget_utilisation_pct']:>7.1f}% "
        f"{m['iterations']:>4} {m['unique_activities']:>6}"
    )
print("="*75)

all_passed = all(m["passed"] for m in all_metrics)
print(f"\nOverall: {'✅ ALL SCENARIOS PASSED' if all_passed else '❌ SOME SCENARIOS FAILED'}")
assert all_passed, "One or more test scenarios did not pass all evaluations!"
print("\n🎉 Test suite complete – all scenarios validated successfully!")